## Landing Zone

## Execution Notes and Batch Logging
This notebook initializes the Landing Zone bucket and top-level prefixes. The logs show the MinIO bucket inventory and every prefix creation so setup reruns can be checked without opening MinIO manually.


**Importing Useful Libraries**

In [1]:
import os
import boto3
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
MINIO_ROLE = "admin"
if MINIO_ROLE == "admin":
    access_key = os.getenv("MINIO_ACCESS_KEY")
    secret_key = os.getenv("MINIO_SECRET_KEY")
else:
    role_prefix = MINIO_ROLE.upper()
    access_key = os.getenv(f"MINIO_{role_prefix}_ACCESS_KEY")
    secret_key = os.getenv(f"MINIO_{role_prefix}_SECRET_KEY")
if not endpoint or not access_key or not secret_key:
    raise RuntimeError(f"Missing MinIO {MINIO_ROLE} credentials in environment")


In [2]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)


In [3]:
# List existing buckets
buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]
print(f"MinIO connection ready. Existing buckets: {buckets}")

# Function that given a name, creates a bucket
def createBucket(name, list_buckets):
    print(f"Checking bucket: {name}")
    if name in list_buckets:
        print(f"Bucket '{name}' already exists!")
    else:
        s3.create_bucket(Bucket=name)
        print(f"Created bucket: {name}")

# Create a bucket named landing_zone
createBucket("landing-zone", buckets)
buckets_after = [b["Name"] for b in s3.list_buckets()["Buckets"]]
print(f"Landing Zone bucket check complete. Buckets now: {buckets_after}")

MinIO connection ready. Existing buckets: ['a-bucket', 'exploitation-zone', 'landing-zone', 'trusted-zone']
Checking bucket: landing-zone
Bucket 'landing-zone' already exists!
Landing Zone bucket check complete. Buckets now: ['a-bucket', 'exploitation-zone', 'landing-zone', 'trusted-zone']


In [4]:
# Create two sub-buckets inside landing_zone.
for prefix in ["temporal-landing/", "persistent-landing/"]:
    s3.put_object(Bucket="landing-zone", Key=prefix)
    print(f"Created/verified landing prefix: s3://landing-zone/{prefix}")

Created/verified landing prefix: s3://landing-zone/temporal-landing/
Created/verified landing prefix: s3://landing-zone/persistent-landing/
